Dostupni driver za cudu

In [ ]:
!nvidia-smi


Sun Feb  8 02:49:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Naivna verzija Cholesky implementacije CUDA

In [19]:
%%writefile cholesky_v0.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <vector>
#include <random>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do {                                  \
  cudaError_t err = (call);                                    \
  if (err != cudaSuccess) {                                    \
    fprintf(stderr, "CUDA error %s:%d: %s\n",                  \
            __FILE__, __LINE__, cudaGetErrorString(err));      \
    std::exit(1);                                              \
  }                                                            \
} while(0)

// 1) Dijagonalni korak: Lkk = sqrt(Akk - sum_{p<k} Lkp^2)
__global__ void chol_diag_kernel(double* A, int n, int k) {
    if (blockIdx.x == 0 && threadIdx.x == 0) {
        double sum = 0.0;
        for (int p = 0; p < k; p++) {
            double v = A[k + p * n]; // L(k,p)
            sum += v * v;
        }
        double v = A[k + k * n] - sum;
        // SPD bi trebalo da daje v>0; ako numerički ode malo ispod 0, clamp.
        double Lkk = (v > 0.0) ? sqrt(v) : 0.0;
        A[k + k * n] = Lkk;
    }
}

// 2) Kolona ispod dijagonale: Lik = (Aik - sum_{p<k} Lip*Lkp) / Lkk
__global__ void chol_col_kernel(double* A, int n, int k) {
    int i = k + 1 + blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        double sum = 0.0;
        for (int p = 0; p < k; p++) {
            sum += A[i + p * n] * A[k + p * n]; // L(i,p)*L(k,p)
        }
        double Lkk = A[k + k * n];
        A[i + k * n] = (A[i + k * n] - sum) / Lkk;
    }
}

// 3) Update trailing submatrix: Aij -= Lik * Ljk  (samo donji trougao i>=j)
__global__ void chol_update_kernel(double* A, int n, int k) {
    int j = (k + 1) + blockIdx.x * blockDim.x + threadIdx.x;
    int i = (k + 1) + blockIdx.y * blockDim.y + threadIdx.y;

    if (i < n && j < n && i >= j) {
        double Lik = A[i + k * n];
        double Ljk = A[j + k * n];
        A[i + j * n] -= Lik * Ljk;
    }
}

static void make_spd(std::vector<double>& A, int n, unsigned seed=1) {
    // A = M^T M + n I  (SPD)
    std::mt19937 gen(seed);
    std::normal_distribution<double> dist(0.0, 1.0);

    std::vector<double> M(n*n);
    for (int i=0;i<n*n;i++) M[i] = dist(gen);

    // Kolona-major (Fortran): A[i + j*n]
    for (int j=0;j<n;j++) {
        for (int i=0;i<n;i++) {
            double s = 0.0;
            for (int p=0;p<n;p++) {
                double Mpi = M[p + i*n];
                double Mpj = M[p + j*n];
                s += Mpi * Mpj;
            }
            A[i + j*n] = s;
        }
    }
    for (int i=0;i<n;i++) A[i + i*n] += (double)n;
}

static double frob_norm(const std::vector<double>& A) {
    double s=0.0;
    for (double v: A) s += v*v;
    return std::sqrt(s);
}

// Relativni rezidual: ||A - L L^T||_F / ||A||_F
static double rel_residual(const std::vector<double>& Aorig,
                           const std::vector<double>& Afact, int n) {
    std::vector<double> R(n*n, 0.0);

    for (int j=0;j<n;j++) {
        for (int i=0;i<n;i++) {
            double s = 0.0;
            int pmax = (i < j) ? i : j;
            for (int p=0;p<=pmax;p++) {
                double Lip = (i>=p) ? Afact[i + p*n] : 0.0; // L(i,p)
                double Ljp = (j>=p) ? Afact[j + p*n] : 0.0; // L(j,p)
                s += Lip * Ljp;
            }
            R[i + j*n] = Aorig[i + j*n] - s;
        }
    }
    return frob_norm(R) / frob_norm(Aorig);
}

int main() {
    int n = 3000;
    int reps = 3;

    printf("Cholesky v0 (correct) CUDA, n=%d\n", n);

    std::vector<double> A(n*n);
    make_spd(A, n, 42);
    std::vector<double> Aorig = A;

    double* dA = nullptr;
    CUDA_CHECK(cudaMalloc((void**)&dA, sizeof(double)*n*n));

    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    float total_ms = 0.0f;

    for (int r=0;r<reps;r++) {
        CUDA_CHECK(cudaMemcpy(dA, Aorig.data(), sizeof(double)*n*n, cudaMemcpyHostToDevice));

        CUDA_CHECK(cudaEventRecord(start));

        for (int k=0;k<n;k++) {
            // 1) dijagonala (1 thread)
            chol_diag_kernel<<<1,1>>>(dA, n, k);
            CUDA_CHECK(cudaGetLastError());

            int remaining = n - (k + 1);
            if (remaining > 0) {
                // 2) kolona (paralelno po i)
                dim3 block1(256);
                int grid1 = (remaining + block1.x - 1) / block1.x;
                chol_col_kernel<<<grid1, block1>>>(dA, n, k);
                CUDA_CHECK(cudaGetLastError());

                // 3) update trailing (2D grid)
                dim3 block2(16, 16);
                dim3 grid2((remaining + block2.x - 1)/block2.x,
                           (remaining + block2.y - 1)/block2.y);
                chol_update_kernel<<<grid2, block2>>>(dA, n, k);
                CUDA_CHECK(cudaGetLastError());
            }
        }

        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));

        float ms = 0.0f;
        CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
        total_ms += ms;
    }

    float avg_ms = total_ms / reps;
    printf("Avg GPU time: %.3f ms\n", avg_ms);

    CUDA_CHECK(cudaMemcpy(A.data(), dA, sizeof(double)*n*n, cudaMemcpyDeviceToHost));
    printf("Rel residual ||A-LL^T||_F/||A||_F = %.3e\n", rel_residual(Aorig, A, n));

    CUDA_CHECK(cudaFree(dA));
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    return 0;
}


Writing cholesky_v0.cu


In [ ]:
!nvcc -O3 cholesky_v0.cu -o cholesky_v0 -arch=sm_75


In [ ]:
!./cholesky_v0


Cholesky v0 (correct) CUDA, n=3000
Avg GPU time: 1155.160 ms
Rel residual ||A-LL^T||_F/||A||_F = 1.474e-01


Dobijeni je veliki rezidual zbog neblokovkse implementacije i velikog broja uzastopnih kernel poziva

# Blokovski Cholesky- 1

In [ ]:
%%writefile cholesky_blocked.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <vector>
#include <random>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do {                                  \
  cudaError_t err = (call);                                    \
  if (err != cudaSuccess) {                                    \
    fprintf(stderr, "CUDA error %s:%d: %s\n",                  \
            __FILE__, __LINE__, cudaGetErrorString(err));      \
    std::exit(1);                                              \
  }                                                            \
} while(0)


#ifndef B
#define B 32
#endif



__device__ __forceinline__ double& Aat(double* A, int n, int i, int j) {
    return A[i + j*n];
}


__global__ void potrf_tile(double* A, int n, int k) {
    __shared__ double T[B][B+1];

    int tx = threadIdx.x;
    int ty = threadIdx.y;

    int i = k + ty;
    int j = k + tx;
    if (i < n && j < n) T[ty][tx] = Aat(A, n, i, j);
    else T[ty][tx] = 0.0;
    __syncthreads();


    for (int p = 0; p < B; p++) {
        if (ty == p && tx == p) {
            double sum = 0.0;
            for (int s = 0; s < p; s++) sum += T[p][s] * T[p][s];
            double v = T[p][p] - sum;
            T[p][p] = (v > 0.0) ? sqrt(v) : 0.0;
        }


        __syncthreads();



     // T[r][p] /= T[p][p]
        if (tx == p && ty > p) {
            double sum = 0.0;
            for (int s = 0; s < p; s++) sum += T[ty][s] * T[p][s];
            T[ty][p] = (T[ty][p] - sum) / T[p][p];
        }
        __syncthreads();


        if (ty < tx) T[ty][tx] = 0.0;
        __syncthreads();
    }

    if (i < n && j < n) Aat(A, n, i, j) = T[ty][tx];
}




__global__ void trsm_tile(double* A, int n, int k) {
    int tile_i = k + B + blockIdx.x * B;

    __shared__ double Lkk[B][B+1];
    __shared__ double Lik[B][B+1];

    int tx = threadIdx.x;
    int ty = threadIdx.y;

    int i = k + ty;
    int j = k + tx;
    Lkk[ty][tx] = (i < n && j < n) ? Aat(A, n, i, j) : 0.0;


    int ri = tile_i + ty;
    int cj = k + tx;
    Lik[ty][tx] = (ri < n && cj < n) ? Aat(A, n, ri, cj) : 0.0;
    __syncthreads();


    // Lik(r,p) = (Lik(r,p) - sum_{s<p} Lik(r,s)*Lkk(p,s)) / Lkk(p,p)
    for (int p = 0; p < B; p++) {
        if (ty < B && tx == p) {
            double sum = 0.0;
            for (int s = 0; s < p; s++) sum += Lik[ty][s] * Lkk[p][s];
            Lik[ty][p] = (Lik[ty][p] - sum) / Lkk[p][p];
        }
        __syncthreads();
    }

    if (ri < n && cj < n) Aat(A, n, ri, cj) = Lik[ty][tx];
}




//Ažuriranje donjeg desnog dijela matrice
__global__ void gemm_update(double* A, int n, int k) {

    int bi = blockIdx.y;
    int bj = blockIdx.x;

    int i0 = k + B + bi * B;
    int j0 = k + B + bj * B;

    if (j0 > i0) return;

    __shared__ double Lik[B][B+1];
    __shared__ double Ljk[B][B+1];

    int tx = threadIdx.x;
    int ty = threadIdx.y;

    int i = i0 + ty;
    int j = j0 + tx;


    int colk = k + tx;
    Lik[ty][tx] = (i < n && colk < n) ? Aat(A, n, i, colk) : 0.0;

    int jr = j0 + ty;
    Ljk[ty][tx] = (jr < n && colk < n) ? Aat(A, n, jr, colk) : 0.0;

    __syncthreads();

    if (i < n && j < n && i >= j) {
        double sum = 0.0;
        #pragma unroll
        for (int p = 0; p < B; p++) {
            sum += Lik[ty][p] * Ljk[tx][p];
        }
        Aat(A, n, i, j) -= sum;
    }
}






static void make_spd(std::vector<double>& A, int n, unsigned seed=1) {
    std::mt19937 gen(seed);
    std::normal_distribution<double> dist(0.0, 1.0);

    std::vector<double> M(n*n);
    for (int i=0;i<n*n;i++) M[i] = dist(gen);

    for (int j=0;j<n;j++) {
        for (int i=0;i<n;i++) {
            double s = 0.0;
            for (int p=0;p<n;p++) {
                double Mpi = M[p + i*n];
                double Mpj = M[p + j*n];
                s += Mpi * Mpj;
            }
            A[i + j*n] = s;
        }
    }
    for (int i=0;i<n;i++) A[i + i*n] += (double)n;
}

static double frob_norm(const std::vector<double>& A) {
    double s=0.0;
    for (double v: A) s += v*v;
    return std::sqrt(s);
}

static double rel_residual(const std::vector<double>& Aorig,
                           const std::vector<double>& Afact, int n) {
    std::vector<double> R(n*n, 0.0);

    for (int j=0;j<n;j++) {
        for (int i=0;i<n;i++) {
            double s = 0.0;
            int pmax = (i < j) ? i : j;
            for (int p=0;p<=pmax;p++) {
                double Lip = (i>=p) ? Afact[i + p*n] : 0.0;
                double Ljp = (j>=p) ? Afact[j + p*n] : 0.0;
                s += Lip * Ljp;
            }
            R[i + j*n] = Aorig[i + j*n] - s;
        }
    }
    return frob_norm(R) / frob_norm(Aorig);
}

int main() {
    int n = 512;
    int reps = 3;

    printf("Blocked Cholesky (tile B=%d), n=%d\n", B, n);

    std::vector<double> A(n*n);
    make_spd(A, n, 42);
    std::vector<double> Aorig = A;

    double* dA = nullptr;
    CUDA_CHECK(cudaMalloc((void**)&dA, sizeof(double)*n*n));

    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    float total_ms = 0.0f;

    dim3 block2(B, B);

    for (int r=0;r<reps;r++) {
        CUDA_CHECK(cudaMemcpy(dA, Aorig.data(), sizeof(double)*n*n, cudaMemcpyHostToDevice));

        CUDA_CHECK(cudaEventRecord(start));

        for (int k = 0; k < n; k += B) {
            potrf_tile<<<1, block2>>>(dA, n, k);

            int tiles_below = (n - (k + B) + B - 1) / B;
            if (tiles_below > 0) {
                trsm_tile<<<tiles_below, block2>>>(dA, n, k);
            }

            int tiles_trailing = (n - (k + B) + B - 1) / B;
            if (tiles_trailing > 0) {
                dim3 grid(tiles_trailing, tiles_trailing);
                gemm_update<<<grid, block2>>>(dA, n, k);
            }
        }

        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));

        float ms = 0.0f;
        CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
        total_ms += ms;
    }

    printf("Avg GPU time: %.3f ms\n", total_ms / reps);

    CUDA_CHECK(cudaMemcpy(A.data(), dA, sizeof(double)*n*n, cudaMemcpyDeviceToHost));
    printf("Rel residual ||A-LL^T||_F/||A||_F = %.3e\n", rel_residual(Aorig, A, n));

    CUDA_CHECK(cudaFree(dA));
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    return 0;
}


Writing cholesky_blocked.cu


In [ ]:
!nvcc -O3 -arch=sm_75 cholesky_blocked.cu -o chol_blocked


In [ ]:
!./chol_blocked


Blocked Cholesky (tile B=32), n=512
Avg GPU time: 19.264 ms
Rel residual ||A-LL^T||_F/||A||_F = 1.675e-16


In [ ]:
!nvcc -O3 -DB=16 cholesky_blocked.cu -o chol_B16
!nvcc -O3 -DB=32 cholesky_blocked.cu -o chol_B32
!nvcc -O3 -DB=64 cholesky_blocked.cu -o chol_B64


In [ ]:
!./chol_B16
!./chol_B32
!./chol_B64


Blocked Cholesky (tile B=16), n=512
Avg GPU time: 15.201 ms
Rel residual ||A-LL^T||_F/||A||_F = 1.170e+03
Blocked Cholesky (tile B=32), n=512
Avg GPU time: 2.405 ms
Rel residual ||A-LL^T||_F/||A||_F = 1.170e+03
Blocked Cholesky (tile B=64), n=512
Avg GPU time: 2.417 ms
Rel residual ||A-LL^T||_F/||A||_F = 1.170e+03


# Blokovski Cholesky -2

In [ ]:
%%writefile cholesky_gpu_pro.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>

#define B 16

#define CUDA_CHECK(call) do {                                  \
  cudaError_t err = (call);                                    \
  if (err != cudaSuccess) {                                    \
    fprintf(stderr, "CUDA error %s:%d: %s\n",                  \
            __FILE__, __LINE__, cudaGetErrorString(err));      \
    exit(1);                                                   \
  }                                                            \
} while(0)



inline void checkKernel(const char* label) {
    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess) {
        printf("Greška u %s: %s\n", label, cudaGetErrorString(err));
        exit(1);
    }
    cudaDeviceSynchronize();
}




__device__ __forceinline__ double& Aat(double* A, int n, int i, int j) {
    return A[i + j * n];
}



__global__ void potrf_kernel(double* A, int n, int k) {
    __shared__ double tile[B][B + 1];

    int tx = threadIdx.x;
    int ty = threadIdx.y;
    int i = k + ty;
    int j = k + tx;

    if (i < n && j < n) tile[ty][tx] = Aat(A, n, i, j);
    else tile[ty][tx] = 0.0;
    __syncthreads();

    for (int p = 0; p < B; p++) {
        double diag_val = sqrt(tile[p][p]);
        if (ty == p && tx == p) tile[p][p] = diag_val;
        __syncthreads();

        if (ty > p && tx == p) tile[ty][p] /= diag_val;
        __syncthreads();

        if (ty > p && tx > p && tx <= ty) {
            tile[ty][tx] -= tile[ty][p] * tile[tx][p];
        }
        __syncthreads();
    }

    if (i < n && j < n && i >= j) Aat(A, n, i, j) = tile[ty][tx];
    else if (i < n && j < n && i < j) Aat(A, n, i, j) = 0.0;
}



__global__ void trsm_kernel(double* A, int n, int k) {
    int tile_idx = blockIdx.x + 1;
    int i_base = k + tile_idx * B;
    if (i_base >= n) return;

    __shared__ double L_diag[B][B + 1];
    __shared__ double A_block[B][B + 1];

    int tx = threadIdx.x;
    int ty = threadIdx.y;

    L_diag[ty][tx] = ( (k+ty) < n && (k+tx) < n ) ? Aat(A, n, k+ty, k+tx) : 0.0;
    A_block[ty][tx] = ( (i_base+ty) < n && (k+tx) < n ) ? Aat(A, n, i_base+ty, k+tx) : 0.0;
    __syncthreads();

    for (int p = 0; p < B; p++) {
        if (tx > p) A_block[ty][tx] -= A_block[ty][p] * L_diag[tx][p];
        __syncthreads();
        if (tx == p) A_block[ty][p] /= L_diag[p][p];
        __syncthreads();
    }

    if ((i_base + ty) < n && (k + tx) < n) Aat(A, n, i_base + ty, k + tx) = A_block[ty][tx];
}




__global__ void gemm_kernel(double* A, int n, int k) {
    int tx_idx = blockIdx.x;
    int ty_idx = blockIdx.y;
    if (ty_idx < tx_idx) return;

    int row_tile = ty_idx + (k/B) + 1;
    int col_tile = tx_idx + (k/B) + 1;

    __shared__ double L_row[B][B + 1];
    __shared__ double L_col[B][B + 1];

    int tx = threadIdx.x;
    int ty = threadIdx.y;
    int i = row_tile * B + ty;
    int j = col_tile * B + tx;

    L_row[ty][tx] = (i < n && (k + tx) < n) ? Aat(A, n, i, k + tx) : 0.0;
    L_col[ty][tx] = (j < n && (k + tx) < n) ? Aat(A, n, j, k + tx) : 0.0;
    __syncthreads();

    if (i < n && j < n && i >= j) {
        double sum = 0.0;
        for (int p = 0; p < B; p++) sum += L_row[ty][p] * L_col[tx][p];
        Aat(A, n, i, j) -= sum;
    }
}




void solve(double* d_A, int n, bool debug) {
    dim3 threads(B, B);
    for (int k = 0; k < n; k += B) {
        potrf_kernel<<<1, threads>>>(d_A, n, k);
        if(debug) checkKernel("POTRF");

        int tiles_left = (n - k - B + B - 1) / B;
        if (tiles_left > 0) {
            trsm_kernel<<<tiles_left, threads>>>(d_A, n, k);
            if(debug) checkKernel("TRSM");

            dim3 grid_gemm(tiles_left, tiles_left);
            gemm_kernel<<<grid_gemm, threads>>>(d_A, n, k);
            if(debug) checkKernel("GEMM");
        }
    }
}

int main() {
    cudaFree(0);
    int n = 3000;
    int iterations = 10;
    size_t size = n * n * sizeof(double);

    double *h_A;
    CUDA_CHECK(cudaHostAlloc(&h_A, size, cudaHostAllocDefault));

    for (int i = 0; i < n * n; i++) h_A[i] = 1.0;
    for (int i = 0; i < n; i++) h_A[i + i*n] = n * 2.0;

    double *d_A;
    CUDA_CHECK(cudaMalloc(&d_A, size));

    CUDA_CHECK(cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice));
    solve(d_A, n, false);
    cudaDeviceSynchronize();

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    float total_ms = 0;

    printf("Pokreće se % d iteracija: \n", iterations);
    for (int i = 0; i < iterations; i++) {
        CUDA_CHECK(cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice));

        cudaEventRecord(start);
        solve(d_A, n, false);
        cudaEventRecord(stop);
        cudaEventSynchronize(stop);

        float ms = 0;
        cudaEventElapsedTime(&ms, start, stop);
        total_ms += ms;
        printf("  Iteracija %d: %.3f ms\n", i + 1, ms);
    }

    printf("\n REZULTATI\n");
    printf("Prosečno vrijeme (%d iteracija): %.3f ms\n", iterations, total_ms / iterations);

    CUDA_CHECK(cudaMemcpy(h_A, d_A, size, cudaMemcpyDeviceToHost));
    printf("Provjera: L[0,0] = %.2f (Očekivano: %.2f)\n", h_A[0], sqrt(n * 2.0));

    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    cudaFreeHost(h_A);
    cudaFree(d_A);

    return 0;
}

Writing cholesky_gpu_pro.cu


In [ ]:
!nvcc -O3 -arch=sm_75 cholesky_gpu_pro.cu -o cholesky_gpu
!./cholesky_gpu

Pokreće se  10 iteracija: 
  Iteracija 1: 79.887 ms
  Iteracija 2: 78.497 ms
  Iteracija 3: 67.921 ms
  Iteracija 4: 63.978 ms
  Iteracija 5: 64.505 ms
  Iteracija 6: 63.914 ms
  Iteracija 7: 65.233 ms
  Iteracija 8: 63.590 ms
  Iteracija 9: 65.362 ms
  Iteracija 10: 64.046 ms

 REZULTATI
Prosečno vrijeme (10 iteracija): 67.693 ms
Provjera: L[0,0] = 77.46 (Očekivano: 77.46)


# BlokovskI Cholesky 3-KONAČNA VERZIJA

In [5]:
%%writefile cholesky_tiled_potrf.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>

#define B 16

#define CUDA_CHECK(call) do {                                  \
  cudaError_t err = (call);                                    \
  if (err != cudaSuccess) {                                    \
    fprintf(stderr, "CUDA error %s:%d: %s\n",                  \
            __FILE__, __LINE__, cudaGetErrorString(err));      \
    exit(1);                                                   \
  }                                                            \
} while(0)

__device__ __forceinline__ double& Aat(double* A, int n, int i, int j) {
    return A[i + j * n];
}

__global__ void potrf_kernel(double* A, int n, int k) {
    __shared__ double tile[B][B + 1];
    int tx = threadIdx.x; int ty = threadIdx.y;
    int i = k + ty; int j = k + tx;

    if (i < n && j < n) tile[ty][tx] = Aat(A, n, i, j);
    else tile[ty][tx] = 0.0;
    __syncthreads();

    for (int p = 0; p < B; p++) {
        if (ty == p && tx == p) {
            double val = tile[p][p];
            if (val < 1e-12) val = 1e-12;
            tile[p][p] = sqrt(val);
        }
        __syncthreads();

        double d = tile[p][p];

        if (ty > p && tx == p) {
            tile[ty][p] /= d;
        }
        __syncthreads();

        if (ty > p && tx >= p && tx <= ty) {
            tile[ty][tx] -= tile[ty][p] * tile[tx][p];
        }
        __syncthreads();
    }

    if (i < n && j < n) {
        if (i >= j) Aat(A, n, i, j) = tile[ty][tx];
        else Aat(A, n, i, j) = 0.0;
    }
}

__global__ void trsm_kernel(double* A, int n, int k) {
    int tile_idx = blockIdx.x + 1;
    int i_base = k + tile_idx * B;
    if (i_base >= n) return;

    __shared__ double L_diag[B][B + 1];
    __shared__ double A_block[B][B + 1];
    int tx = threadIdx.x; int ty = threadIdx.y;

    L_diag[ty][tx] = ((k+ty) < n && (k+tx) < n) ? Aat(A, n, k+ty, k+tx) : 0.0;
    A_block[ty][tx] = ((i_base+ty) < n && (k+tx) < n) ? Aat(A, n, i_base+ty, k+tx) : 0.0;
    __syncthreads();

    for (int p = 0; p < B; p++) {
        if (tx == p) {
            double diag = L_diag[p][p];
            if (fabs(diag) > 1e-12) {
                A_block[ty][p] /= diag;
            }
        }
        __syncthreads();

        if (tx > p) {
            A_block[ty][tx] -= A_block[ty][p] * L_diag[tx][p];
        }
        __syncthreads();
    }

    if ((i_base + ty) < n && (k + tx) < n) Aat(A, n, i_base + ty, k + tx) = A_block[ty][tx];
}

__global__ void gemm_kernel(double* A, int n, int k) {
    int tx_idx = blockIdx.x; int ty_idx = blockIdx.y;
    if (ty_idx < tx_idx) return;

    int row_tile = ty_idx + (k/B) + 1;
    int col_tile = tx_idx + (k/B) + 1;

    __shared__ double L_row[B][B + 1];
    __shared__ double L_col[B][B + 1];
    int tx = threadIdx.x; int ty = threadIdx.y;
    int i = row_tile * B + ty; int j = col_tile * B + tx;

    L_row[ty][tx] = (i < n && (k + tx) < n) ? Aat(A, n, i, k + tx) : 0.0;
    L_col[ty][tx] = (j < n && (k + tx) < n) ? Aat(A, n, j, k + tx) : 0.0;
    __syncthreads();

    if (i < n && j < n && i >= j) {
        double sum = 0.0;
        for (int p = 0; p < B; p++) sum += L_row[ty][p] * L_col[tx][p];
        Aat(A, n, i, j) -= sum;
    }
}

void solve_gpu(double* d_A, int n) {
    dim3 threads(B, B);
    for (int k = 0; k < n; k += B) {
        potrf_kernel<<<1, threads>>>(d_A, n, k);
        int tiles_left = (n - k - B + B - 1) / B;
        if (tiles_left > 0) {
            trsm_kernel<<<tiles_left, threads>>>(d_A, n, k);
            dim3 grid_gemm(tiles_left, tiles_left);
            gemm_kernel<<<grid_gemm, threads>>>(d_A, n, k);
        }
    }
}

void verify(double* h_A_orig, double* h_L, int n) {
    printf("\nVerifikacija tačnosti:\n");
    double max_err = 0;
    double norm_A = 0;

    for (int i = 0; i < n; i++) {
        for (int j = 0; j <= i; j++) {
            norm_A += h_A_orig[i + j * n] * h_A_orig[i + j * n];

            double sum = 0;
            for (int k = 0; k <= j; k++) {
                sum += h_L[i + k * n] * h_L[j + k * n];
            }
            double diff = fabs(sum - h_A_orig[i + j * n]);
            if (diff > max_err) max_err = diff;
        }
    }

    norm_A = sqrt(norm_A);
    double relative_err = max_err / norm_A;

    printf("Maksimalna apsolutna greška: %e\n", max_err);
    printf("Relativna greška: %e\n", relative_err);
    if (relative_err < 1e-6) printf("STATUS: TAČNO (Success)\n");
    else printf("STATUS: GREŠKA JE PREVELIKA (Fail)\n");
}

int main() {
    cudaFree(0);
    int n = 1024;
    int iterations = 10;
    size_t size = (size_t)n * n * sizeof(double);

    double *h_A, *h_A_orig, *d_A;
    CUDA_CHECK(cudaHostAlloc(&h_A, size, cudaHostAllocDefault));
    h_A_orig = (double*)malloc(size);
    CUDA_CHECK(cudaMalloc(&d_A, size));

    for (int j = 0; j < n; j++) {
        for (int i = 0; i < n; i++) {
            if (i == j) {
                h_A[i + j * n] = (double)n + 1.0;
            } else {
                h_A[i + j * n] = 1.0 / (double)n;
            }
        }
    }
    memcpy(h_A_orig, h_A, size);

    double flops = (1.0/3.0) * pow(n, 3);
    double total_bytes = (double)n * n * n / (3.0 * B) * 4.0 * sizeof(double);

    CUDA_CHECK(cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice));
    solve_gpu(d_A, n);
    cudaDeviceSynchronize();

    cudaEvent_t start, stop;
    cudaEventCreate(&start); cudaEventCreate(&stop);

    float total_ms = 0;
    for (int i = 0; i < iterations; i++) {
        CUDA_CHECK(cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice));
        cudaEventRecord(start);
        solve_gpu(d_A, n);
        cudaEventRecord(stop);
        cudaEventSynchronize(stop);
        float ms;
        cudaEventElapsedTime(&ms, start, stop);
        total_ms += ms;
    }

    float avg_ms = total_ms / iterations;
    printf("\nREZULTATI (N=%d)\n", n);
    printf("Prosječno vrijeme: %.3f ms\n", avg_ms);
    printf("Postignuti GFLOPS: %.2f\n", (flops / (avg_ms/1000.0)) / 1e9);
    printf("Efektivni Bandwidth: %.2f GB/s\n", (total_bytes / (avg_ms/1000.0)) / 1e9);

    CUDA_CHECK(cudaMemcpy(h_A, d_A, size, cudaMemcpyDeviceToHost));
    verify(h_A_orig, h_A, n);

    cudaFreeHost(h_A); free(h_A_orig); cudaFree(d_A);
    return 0;
}

Writing cholesky_tiled_potrf.cu


In [6]:
!nvcc -O3 -arch=sm_75 cholesky_tiled_potrf.cu -o cholesky_final

In [7]:
!./cholesky_final



REZULTATI (N=1024)
Prosječno vrijeme: 6.858 ms
Postignuti GFLOPS: 52.19
Efektivni Bandwidth: 104.38 GB/s

Verifikacija tačnosti:
Maksimalna apsolutna greška: 3.126526e-02
Relativna greška: 9.532090e-07
STATUS: TAČNO (Success)


# Testiranje za različite dimenzije matrice

In [1]:
%%writefile cholesky_testing.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>
#include <vector>

#define B 16

#define CUDA_CHECK(call) do {                                  \
  cudaError_t err = (call);                                    \
  if (err != cudaSuccess) {                                    \
    fprintf(stderr, "CUDA error %s:%d: %s\n",                  \
            __FILE__, __LINE__, cudaGetErrorString(err));      \
    exit(1);                                                   \
  }                                                            \
} while(0)

__device__ __forceinline__ double& Aat(double* A, int n, int i, int j) {
    return A[i + j * n];
}

__global__ void potrf_kernel(double* A, int n, int k) {
    __shared__ double tile[B][B + 1];
    int tx = threadIdx.x; int ty = threadIdx.y;
    int i = k + ty; int j = k + tx;

    if (i < n && j < n) tile[ty][tx] = Aat(A, n, i, j);
    else tile[ty][tx] = 0.0;
    __syncthreads();

    for (int p = 0; p < B; p++) {
        if (ty == p && tx == p) {
            double val = tile[p][p];
            if (val < 1e-12) val = 1e-12;
            tile[p][p] = sqrt(val);
        }
        __syncthreads();
        double d = tile[p][p];
        if (ty > p && tx == p) tile[ty][p] /= d;
        __syncthreads();
        if (ty > p && tx >= p && tx <= ty) {
            tile[ty][tx] -= tile[ty][p] * tile[tx][p];
        }
        __syncthreads();
    }
    if (i < n && j < n) {
        if (i >= j) Aat(A, n, i, j) = tile[ty][tx];
        else Aat(A, n, i, j) = 0.0;
    }
}

__global__ void trsm_kernel(double* A, int n, int k) {
    int tile_idx = blockIdx.x + 1;
    int i_base = k + tile_idx * B;
    if (i_base >= n) return;
    __shared__ double L_diag[B][B + 1];
    __shared__ double A_block[B][B + 1];
    int tx = threadIdx.x; int ty = threadIdx.y;
    L_diag[ty][tx] = ((k+ty) < n && (k+tx) < n) ? Aat(A, n, k+ty, k+tx) : 0.0;
    A_block[ty][tx] = ((i_base+ty) < n && (k+tx) < n) ? Aat(A, n, i_base+ty, k+tx) : 0.0;
    __syncthreads();
    for (int p = 0; p < B; p++) {
        if (tx == p) {
            double diag = L_diag[p][p];
            if (fabs(diag) > 1e-12) A_block[ty][p] /= diag;
        }
        __syncthreads();
        if (tx > p) A_block[ty][tx] -= A_block[ty][p] * L_diag[tx][p];
        __syncthreads();
    }
    if ((i_base + ty) < n && (k + tx) < n) Aat(A, n, i_base + ty, k + tx) = A_block[ty][tx];
}

__global__ void gemm_kernel(double* A, int n, int k) {
    int tx_idx = blockIdx.x; int ty_idx = blockIdx.y;
    if (ty_idx < tx_idx) return;
    int row_tile = ty_idx + (k/B) + 1;
    int col_tile = tx_idx + (k/B) + 1;
    __shared__ double L_row[B][B + 1];
    __shared__ double L_col[B][B + 1];
    int tx = threadIdx.x; int ty = threadIdx.y;
    int i = row_tile * B + ty; int j = col_tile * B + tx;
    L_row[ty][tx] = (i < n && (k + tx) < n) ? Aat(A, n, i, k + tx) : 0.0;
    L_col[ty][tx] = (j < n && (k + tx) < n) ? Aat(A, n, j, k + tx) : 0.0;
    __syncthreads();
    if (i < n && j < n && i >= j) {
        double sum = 0.0;
        for (int p = 0; p < B; p++) sum += L_row[ty][p] * L_col[tx][p];
        Aat(A, n, i, j) -= sum;
    }
}

void solve_gpu(double* d_A, int n) {
    dim3 threads(B, B);
    for (int k = 0; k < n; k += B) {
        potrf_kernel<<<1, threads>>>(d_A, n, k);
        int tiles_left = (n - k - B + B - 1) / B;
        if (tiles_left > 0) {
            trsm_kernel<<<tiles_left, threads>>>(d_A, n, k);
            dim3 grid_gemm(tiles_left, tiles_left);
            gemm_kernel<<<grid_gemm, threads>>>(d_A, n, k);
        }
    }
}

int main() {
    std::vector<int> sizes = {256, 512, 1024, 2048, 4096, 8192, 16384};
    int num_runs = 10;

    printf("\n%-10s | %-15s | %-10s\n", "N", "Avg Time (ms)", "L[0,0] OK?");
    printf("-------------------------------------------\n");

    for (int n : sizes) {
        size_t size = (size_t)n * n * sizeof(double);
        double *h_A, *d_A;

        CUDA_CHECK(cudaMallocHost(&h_A, size));
        CUDA_CHECK(cudaMalloc(&d_A, size));

        for (int i = 0; i < n*n; i++) h_A[i] = 1.0/n;
        for (int i = 0; i < n; i++) h_A[i + i*n] = (double)n + 1.0;

        cudaEvent_t start, stop;
        CUDA_CHECK(cudaEventCreate(&start));
        CUDA_CHECK(cudaEventCreate(&stop));

        CUDA_CHECK(cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice));
        solve_gpu(d_A, n);
        CUDA_CHECK(cudaDeviceSynchronize());

        float total_time = 0.0f;
        for (int run = 0; run < num_runs; run++) {
            CUDA_CHECK(cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice));

            CUDA_CHECK(cudaEventRecord(start, 0));
            solve_gpu(d_A, n);
            CUDA_CHECK(cudaEventRecord(stop, 0));
            CUDA_CHECK(cudaEventSynchronize(stop));

            float ms;
            CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
            total_time += ms;
        }

        float avg_time = total_time / num_runs;
        double actual;
        CUDA_CHECK(cudaMemcpy(&actual, d_A, sizeof(double), cudaMemcpyDeviceToHost));
        bool ok = fabs(actual - sqrt((double)n + 1.0)) < 1e-7;

        printf("%-10d | %-15.3f | %-10s\n", n, avg_time, ok ? "DA" : "NE");

        CUDA_CHECK(cudaFreeHost(h_A));
        CUDA_CHECK(cudaFree(d_A));
        CUDA_CHECK(cudaEventDestroy(start));
        CUDA_CHECK(cudaEventDestroy(stop));
    }
    return 0;
}

Writing cholesky_testing.cu


In [2]:
!nvcc -O3 -arch=sm_75 cholesky_testing.cu -o cholesky_testing
!./cholesky_testing


N          | Avg Time (ms)   | L[0,0] OK?
-------------------------------------------
256        | 1.057           | DA        
512        | 2.405           | DA        
1024       | 7.356           | DA        
2048       | 21.922          | DA        
4096       | 151.864         | DA        
8192       | 1156.859        | DA        
16384      | 9707.389        | DA        


# cuBLas pozivanje

In [3]:
%%writefile cholesky_cublas_final.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <vector>
#include <cuda_runtime.h>
#include <cusolverDn.h>

#define CUDA_CHECK(call) do {                                  \
  cudaError_t err = (call);                                    \
  if (err != cudaSuccess) {                                    \
    fprintf(stderr, "CUDA error %s:%d: %s\n",                  \
            __FILE__, __LINE__, cudaGetErrorString(err));      \
    exit(1);                                                   \
  }                                                            \
} while(0)

#define CUSOLVER_CHECK(call) do {                              \
  cusolverStatus_t status = (call);                            \
  if (status != CUSOLVER_STATUS_SUCCESS) {                     \
    fprintf(stderr, "cuSOLVER error %s:%d: code=%d\n",         \
            __FILE__, __LINE__, (int)status);                  \
    exit(1);                                                   \
  }                                                            \
} while(0)

int main() {
    std::vector<int> sizes = {256, 512, 1024, 2048, 4096, 8192, 16384};
    int num_runs = 10;

    printf("\n%-10s | %-15s | %-10s\n", "N", "Avg Time (ms)", "Status");
    printf("-------------------------------------------\n");

    cusolverDnHandle_t handle;
    CUSOLVER_CHECK(cusolverDnCreate(&handle));

    for (int n : sizes) {
        size_t size = (size_t)n * n * sizeof(double);
        double *h_A, *d_A, *d_workspace;
        int *d_info;

        CUDA_CHECK(cudaMallocHost(&h_A, size));
        CUDA_CHECK(cudaMalloc(&d_A, size));
        CUDA_CHECK(cudaMalloc(&d_info, sizeof(int)));

        for (int i = 0; i < n * n; i++) h_A[i] = 1.0 / n;
        for (int i = 0; i < n; i++) h_A[i + i*n] = (double)n + 1.0;

        int workspace_size = 0;
        CUSOLVER_CHECK(cusolverDnDpotrf_bufferSize(handle, CUBLAS_FILL_MODE_LOWER, n, d_A, n, &workspace_size));
        CUDA_CHECK(cudaMalloc(&d_workspace, sizeof(double) * workspace_size));

        for(int i = 0; i < 2; i++) {
            CUDA_CHECK(cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice));
            CUSOLVER_CHECK(cusolverDnDpotrf(handle, CUBLAS_FILL_MODE_LOWER, n, d_A, n, d_workspace, workspace_size, d_info));
        }
        CUDA_CHECK(cudaDeviceSynchronize());

        cudaEvent_t start, stop;
        CUDA_CHECK(cudaEventCreate(&start));
        CUDA_CHECK(cudaEventCreate(&stop));

        float total_ms = 0;
        for (int run = 0; run < num_runs; run++) {
            CUDA_CHECK(cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice));

            CUDA_CHECK(cudaEventRecord(start));
            CUSOLVER_CHECK(cusolverDnDpotrf(handle, CUBLAS_FILL_MODE_LOWER, n, d_A, n, d_workspace, workspace_size, d_info));
            CUDA_CHECK(cudaEventRecord(stop));

            CUDA_CHECK(cudaEventSynchronize(stop));
            float ms = 0;
            CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
            total_ms += ms;
        }

        float avg_ms = total_ms / num_runs;

        int h_info = 0;
        CUDA_CHECK(cudaMemcpy(&h_info, d_info, sizeof(int), cudaMemcpyDeviceToHost));

        printf("%-10d | %-15.3f | %-10s\n", n, avg_ms, (h_info == 0 ? "OK" : "FAIL"));

        CUDA_CHECK(cudaEventDestroy(start));
        CUDA_CHECK(cudaEventDestroy(stop));
        CUDA_CHECK(cudaFree(d_A));
        CUDA_CHECK(cudaFree(d_workspace));
        CUDA_CHECK(cudaFree(d_info));
        CUDA_CHECK(cudaFreeHost(h_A));
    }

    cusolverDnDestroy(handle);
    return 0;
}

Writing cholesky_cublas_final.cu


In [4]:
!nvcc -O3 -arch=sm_75 cholesky_cublas_final.cu -o cholesky_cublas_final -lcusolver
!./cholesky_cublas_final


N          | Avg Time (ms)   | Status    
-------------------------------------------
256        | 0.664           | OK        
512        | 2.297           | OK        
1024       | 6.668           | OK        
2048       | 18.904          | OK        
4096       | 105.158         | OK        
8192       | 772.152         | OK        
16384      | 5940.560        | OK        


In [20]:
%%writefile cholesky_cublas_1024.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>
#include <cusolverDn.h>

#define CUDA_CHECK(call) do {                                  \
  cudaError_t err = (call);                                    \
  if (err != cudaSuccess) {                                    \
    fprintf(stderr, "CUDA error %s:%d: %s\n",                  \
            __FILE__, __LINE__, cudaGetErrorString(err));      \
    exit(1);                                                   \
  }                                                            \
} while(0)

#define CUSOLVER_CHECK(call) do {                              \
  cusolverStatus_t status = (call);                            \
  if (status != CUSOLVER_STATUS_SUCCESS) {                     \
    fprintf(stderr, "cuSOLVER error %s:%d: code=%d\n",         \
            __FILE__, __LINE__, (int)status);                  \
    exit(1);                                                   \
  }                                                            \
} while(0)

int main() {
    const int n = 1024;
    const int num_runs = 10; // Veći broj iteracija za precizniju analizu
    size_t size = (size_t)n * n * sizeof(double);

    printf("Analiza cuSOLVER performansi za N = %d\n", n);

    double *h_A, *d_A, *d_workspace;
    int *d_info;
    CUDA_CHECK(cudaMallocHost(&h_A, size)); // Pinned memory
    CUDA_CHECK(cudaMalloc(&d_A, size));
    CUDA_CHECK(cudaMalloc(&d_info, sizeof(int)));

    for (int j = 0; j < n; j++) {
        for (int i = 0; i < n; i++) {
            if (i == j) h_A[i + j * n] = (double)n + 1.0;
            else h_A[i + j * n] = 1.0 / (double)n;
        }
    }

    cusolverDnHandle_t handle;
    CUSOLVER_CHECK(cusolverDnCreate(&handle));

    int workspace_size = 0;
    CUSOLVER_CHECK(cusolverDnDpotrf_bufferSize(
        handle, CUBLAS_FILL_MODE_LOWER, n, d_A, n, &workspace_size));
    CUDA_CHECK(cudaMalloc(&d_workspace, sizeof(double) * workspace_size));

    for(int i = 0; i < 3; i++) {
        CUDA_CHECK(cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice));
        CUSOLVER_CHECK(cusolverDnDpotrf(handle, CUBLAS_FILL_MODE_LOWER, n, d_A, n, d_workspace, workspace_size, d_info));
    }
    CUDA_CHECK(cudaDeviceSynchronize());

    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    float total_ms = 0;
    for (int run = 0; run < num_runs; run++) {
        CUDA_CHECK(cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice));

        CUDA_CHECK(cudaEventRecord(start));
        CUSOLVER_CHECK(cusolverDnDpotrf(handle, CUBLAS_FILL_MODE_LOWER, n, d_A, n, d_workspace, workspace_size, d_info));
        CUDA_CHECK(cudaEventRecord(stop));

        CUDA_CHECK(cudaEventSynchronize(stop));
        float ms = 0;
        CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
        total_ms += ms;
    }

    float avg_ms = total_ms / num_runs;

    int h_info = 0;
    CUDA_CHECK(cudaMemcpy(&h_info, d_info, sizeof(int), cudaMemcpyDeviceToHost));

    double actual_L00;
    CUDA_CHECK(cudaMemcpy(&actual_L00, d_A, sizeof(double), cudaMemcpyDeviceToHost));
    double expected_L00 = sqrt((double)n + 1.0);
    bool ok = (h_info == 0) && (fabs(actual_L00 - expected_L00) < 1e-8);

    printf("Prosjecno vrijeme (%d pokretanja): %.4f ms\n", num_runs, avg_ms);
    printf("Status verifikacije: %s\n", ok ? "USPJESNO (OK)" : "GRESKA (FAIL)");

    CUSOLVER_CHECK(cusolverDnDestroy(handle));
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    CUDA_CHECK(cudaFree(d_A));
    CUDA_CHECK(cudaFree(d_workspace));
    CUDA_CHECK(cudaFree(d_info));
    CUDA_CHECK(cudaFreeHost(h_A));

    return 0;
}

Writing cholesky_cublas_1024.cu


In [22]:
!nvcc -O3 -arch=sm_75 cholesky_cublas_1024.cu -o cholesky_cublas_1024 -lcusolver
!./cholesky_cublas_1024

Analiza cuSOLVER performansi za N = 1024
Prosjecno vrijeme (10 pokretanja): 11.3889 ms
Status verifikacije: USPJESNO (OK)


# Spašavanje rezultata za NVIDIA NSIGHT





In [8]:
!which nsys || echo "nsys nije instaliran"
!which ncu  || echo "ncu nije instaliran"
!ls /usr/local/cuda/bin | grep -E "nsys|ncu" || true


nsys nije instaliran
/usr/local/cuda/bin/ncu
ncu
ncu-ui


In [16]:
!ncu --kernel-name trsm_kernel --launch-count 1 --set full -o trsm_detaljno ./cholesky_final

==PROF== Connected to process 31143 (/content/cholesky_final)
==PROF== Profiling "trsm_kernel": 0%....50%....100% - 31 passes

REZULTATI (N=1024)
Prosječno vrijeme: 8.441 ms
Postignuti GFLOPS: 42.40
Efektivni Bandwidth: 84.80 GB/s

Verifikacija tačnosti:
Maksimalna apsolutna greška: 3.126526e-02
Relativna greška: 9.532090e-07
STATUS: TAČNO (Success)
==PROF== Disconnected from process 31143
==PROF== Report: /content/trsm_detaljno.ncu-rep


In [18]:
from google.colab import files
files.download('trsm_detaljno.ncu-rep')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download('cholesky_report.ncu-rep')